In [1]:
# CELL 1
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, classification_report
import warnings

warnings.filterwarnings('ignore')
print("Libraries loaded successfully.")

Libraries loaded successfully.


In [ ]:
# CELL 2: Data Loading, Temporal Duration & Concurrent Network Graphing
import pandas as pd
import numpy as np

file_path = 'Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv'
df = pd.read_csv(file_path)

# 1. Base Temporal
df['start_datetime'] = pd.to_datetime(df['start_datetime'], errors='coerce')
df['closed_datetime'] = pd.to_datetime(df['closed_datetime'], errors='coerce')
df['hour'] = df['start_datetime'].dt.hour
df['day_of_week'] = df['start_datetime'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

# 2. Fill Missing Categoricals
df['description'] = df['description'].fillna('none')
df['event_cause'] = df['event_cause'].fillna('others')
df['priority'] = df['priority'].fillna('Low')
df['corridor'] = df['corridor'].fillna('Non-corridor')
df['zone'] = df['zone'].fillna('Unknown')
df['requires_road_closure'] = df['requires_road_closure'].fillna(False).astype(int)

# 3. Vehicle Bucketing
def map_veh_bucket(v):
    v = str(v).lower()
    if v in ['private_car', 'taxi', 'lcv']: return 'lmv'
    elif v in ['bmtc_bus', 'heavy_vehicle', 'truck', 'private_bus', 'ksrtc_bus']: return 'large_vehicle'
    else: return 'small_or_none'
df['veh_bucket'] = df['veh_type'].apply(map_veh_bucket)

# 4. Calculate Historic Duration Proxy (Clearance Time in Minutes)
df['duration_mins'] = (df['closed_datetime'] - df['start_datetime']).dt.total_seconds() / 60
# Impute missing durations with the median (approx 120 mins) to save data
df['duration_mins'] = df['duration_mins'].fillna(120).clip(lower=10, upper=1440) 

# 5. Network Graphing (Concurrent Active Events in Zone)
print("Calculating Network Effects (Concurrent Events)...")
def get_concurrent(row):
    if pd.isna(row['start_datetime']) or pd.isna(row['zone']): return 0
    # Find how many events in the same zone started BEFORE this one, and CLOSED AFTER this one started
    mask = (df['zone'] == row['zone']) & (df['start_datetime'] <= row['start_datetime']) & (df['closed_datetime'] >= row['start_datetime'])
    return max(0, mask.sum() - 1) # Subtract self

df['concurrent_events'] = df.apply(get_concurrent, axis=1)

print(f"Data Engineering Complete. Shape: {df.shape}")

Calculating Network Effects (Concurrent Events)...
Data Engineering Complete. Shape: (8173, 52)


In [3]:
# CELL 3: 100% Compliant NLP (TF-IDF + Latent Semantic Analysis)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

print("Extracting Compliant Semantic Features (TF-IDF)...")

# 1. TF-IDF learns vocabulary STRICTLY from the provided dataset (No external LLM weights)
tfidf = TfidfVectorizer(max_features=500, stop_words='english', ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df['description'])

# 2. Compress the sparse 500-word matrix into 5 dense Latent Semantic features
svd = TruncatedSVD(n_components=5, random_state=42)
semantic_features = svd.fit_transform(tfidf_matrix)

for i in range(5):
    df[f'semantic_f{i}'] = semantic_features[:, i]

print("Compliant NLP Pipeline Ready.")

Extracting Compliant Semantic Features (TF-IDF)...
Compliant NLP Pipeline Ready.


In [ ]:
# CELL 4: Ground Truth Engineering (TII)
def calculate_tii(row):
    cause = str(row['event_cause']).lower()
    
    # 1. Base Severity
    if cause in ['water_logging', 'tree_fall', 'pot_holes', 'debris', 'accident']: base = 40
    elif cause in ['construction', 'road_conditions', 'vehicle_breakdown', 'fog / low visibility']: base = 20
    else: base = 15
        
    # 2. Vehicle Penalty
    veh_penalty = 15 if row['veh_bucket'] == 'large_vehicle' else (5 if row['veh_bucket'] == 'lmv' else 0)
    
    # 3. Priority Penalty
    priority_score = 15 if str(row['priority']).lower() == 'high' else 0
    
    # 4. Network Effect Penalty (Capped and Filtered)
    if str(row['zone']) == 'Unknown':
        valid_concurrent = 0
    else:
        valid_concurrent = min(row['concurrent_events'], 5)
        
    network_penalty = valid_concurrent * 3  # Max 15 possible penalty points
    
    raw_score = base + priority_score + veh_penalty + network_penalty
    
    # 5. Physics Multipliers
    m_closure = 1.5 if row['requires_road_closure'] == 1 else 1.0
    m_corridor = 1.0 if str(row['corridor']).lower() == 'non-corridor' else 1.2
    m_peak = 1.3 if pd.notna(row['hour']) and ((8 <= row['hour'] <= 11) or (17 <= row['hour'] <= 20)) else 1.0
    
    # 6. Final Calculation
    tii = raw_score * m_closure * m_corridor * m_peak
    return max(0.0, min(100.0, tii))

df['TII'] = df.apply(calculate_tii, axis=1)
print("Traffic Impact Index (TII) Engineered.")
print(f"New Average TII: {df['TII'].mean():.2f}")

Traffic Impact Index (TII) Engineered.
New Average TII: 56.05


In [12]:
# CELL 5 & 6: Training the Dual AI Pipeline
from sklearn.model_selection import train_test_split
import lightgbm as lgb

features = ['event_cause', 'corridor', 'zone', 'hour', 'day_of_week', 'is_weekend',
            'veh_bucket', 'concurrent_events', 
            'semantic_f0', 'semantic_f1', 'semantic_f2', 'semantic_f3', 'semantic_f4']

model_df = df.dropna(subset=features + ['TII', 'requires_road_closure']).copy()

categorical_features = ['event_cause', 'corridor', 'zone', 'veh_bucket']
for col in categorical_features:
    model_df[col] = model_df[col].astype('category')

X = model_df[features]

# --- MODEL 1: Predict TII (Regression) ---
y_tii = model_df['TII']
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(X, y_tii, test_size=0.2, random_state=42)

train_data_t = lgb.Dataset(X_train_t, label=y_train_t)
params_t = {'objective': 'regression', 'metric': 'rmse', 'verbose': -1}
model_tii = lgb.train(params_t, train_data_t, num_boost_round=100)

# --- MODEL 2: Predict Road Closure Probability (Classification) ---
y_closure = model_df['requires_road_closure']
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y_closure, test_size=0.2, random_state=42)

train_data_c = lgb.Dataset(X_train_c, label=y_train_c)
params_c = {'objective': 'binary', 'metric': 'binary_logloss', 'verbose': -1}
model_closure = lgb.train(params_c, train_data_c, num_boost_round=100)

print("Dual Models Trained Successfully.")

Dual Models Trained Successfully.


In [13]:
# CELL 7: Comprehensive AI Pipeline Evaluation
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix

print("="*60)
print("🚦 MODEL 1 EVALUATION: TRAFFIC IMPACT INDEX (REGRESSION)")
print("="*60)
y_pred_tii = model_tii.predict(X_test_t)
rmse = np.sqrt(mean_squared_error(y_test_t, y_pred_tii))
mae = mean_absolute_error(y_test_t, y_pred_tii)
r2 = r2_score(y_test_t, y_pred_tii)

print(f"RMSE (Root Mean Squared Error): {rmse:.4f} TII points")
print(f"MAE (Mean Absolute Error):      {mae:.4f} TII points")
print(f"R-Squared (Explained Variance): {r2:.4f}")
print(f"Average Predicted TII:          {np.mean(y_pred_tii):.2f}")
print(f"Average Actual TII:             {np.mean(y_test_t):.2f}")

print("\n" + "="*60)
print("🚧 MODEL 2 EVALUATION: ROAD CLOSURE (CLASSIFICATION)")
print("="*60)
# Predict probabilities, then convert to 0 or 1 based on a 50% threshold
y_pred_prob_c = model_closure.predict(X_test_c)
y_pred_class_c = (y_pred_prob_c >= 0.5).astype(int)

accuracy = accuracy_score(y_test_c, y_pred_class_c)
roc_auc = roc_auc_score(y_test_c, y_pred_prob_c)

print(f"Accuracy Score: {accuracy:.4f}")
print(f"ROC-AUC Score:  {roc_auc:.4f}\n")

print("--- Classification Report ---")
print(classification_report(y_test_c, y_pred_class_c, target_names=['No Closure', 'Requires Closure']))

print("--- Confusion Matrix ---")
cm = confusion_matrix(y_test_c, y_pred_class_c)
print(f"True Negatives: {cm[0][0]}  |  False Positives: {cm[0][1]}")
print(f"False Negatives: {cm[1][0]} |  True Positives: {cm[1][1]}")
print("="*60)

🚦 MODEL 1 EVALUATION: TRAFFIC IMPACT INDEX (REGRESSION)
RMSE (Root Mean Squared Error): 5.8247 TII points
MAE (Mean Absolute Error):      2.8863 TII points
R-Squared (Explained Variance): 0.9187
Average Predicted TII:          56.83
Average Actual TII:             56.69

🚧 MODEL 2 EVALUATION: ROAD CLOSURE (CLASSIFICATION)
Accuracy Score: 0.9349
ROC-AUC Score:  0.7405

--- Classification Report ---
                  precision    recall  f1-score   support

      No Closure       0.94      0.99      0.97      1506
Requires Closure       0.52      0.15      0.23       106

        accuracy                           0.93      1612
       macro avg       0.73      0.57      0.60      1612
    weighted avg       0.91      0.93      0.92      1612

--- Confusion Matrix ---
True Negatives: 1491  |  False Positives: 15
False Negatives: 90 |  True Positives: 16


In [14]:
# CELL 8: The Live System Prototype
def generate_dispatch_plan(tii_score, closure_prob, clearance_mins):
    # 1. Apply operations safety buffer to TII
    safe_tii = min(100.0, tii_score + 3.0)
    
    # 2. Determine Manpower via Historical Duration Proxy
    if clearance_mins > 180:
        manpower = "HIGH EFFORT (8+ Personnel, Multi-Shift)"
    elif clearance_mins > 60:
        manpower = "MODERATE EFFORT (3-5 Personnel, Standard Patrol)"
    else:
        manpower = "LOW EFFORT (1-2 Personnel, Quick Clearance)"
        
    # 3. Determine Barricades via ML Closure Probability
    if closure_prob >= 0.50:
        barricades = "Tier-1 Barricading (High Closure Probability)"
    else:
        barricades = "No static barricades required"
        
    # 4. Status based on safe_tii
    if safe_tii >= 75: status = "🚨 SEVERE IMPACT"
    elif safe_tii >= 40: status = "⚠️ MODERATE IMPACT"
    else: status = "✅ LOW IMPACT"
        
    return {
        "AI Status": status,
        "Buffered TII": f"{safe_tii:.1f} / 100",
        "Manpower": manpower,
        "Barricades": barricades
    }

# --- TEST THE SYSTEM ON A MOCK DISPATCH ---
sample_idx = 42 # Grab a random row from the test set
test_case = X_test_t.iloc[[sample_idx]]

# 1. ML Predictions
sample_tii = model_tii.predict(test_case)[0]
sample_closure_prob = model_closure.predict(test_case)[0]

# 2. Query Historical Database for Duration
cause = test_case['event_cause'].values[0]
corridor = test_case['corridor'].values[0]
mask = (df['event_cause'] == cause) & (df['corridor'] == corridor)
hist_durations = df[mask]['duration_mins']
median_mins = hist_durations.median() if not hist_durations.empty else 120

# 3. Generate Plan
plan = generate_dispatch_plan(sample_tii, sample_closure_prob, median_mins)

print("--- RAW DAY-0 INTELLIGENCE ---")
print(f"Cause:              {cause}")
print(f"Corridor:           {corridor}")
print(f"Active Network:     {test_case['concurrent_events'].values[0]} other events nearby")
print("\n--- AI PREDICTIONS ---")
print(f"Predicted TII:      {sample_tii:.1f}/100")
print(f"Closure Risk:       {sample_closure_prob*100:.1f}%")
print(f"Historic Clearance: {median_mins:.0f} mins")
print("\n--- FINAL DISPATCH PLAN ---")
for key, value in plan.items():
    print(f"{key:<15}: {value}")

--- RAW DAY-0 INTELLIGENCE ---
Cause:              vehicle_breakdown
Corridor:           Non-corridor
Active Network:     102 other events nearby

--- AI PREDICTIONS ---
Predicted TII:      37.1/100
Closure Risk:       12.1%
Historic Clearance: 120 mins

--- FINAL DISPATCH PLAN ---
AI Status      : ⚠️ MODERATE IMPACT
Buffered TII   : 40.1 / 100
Manpower       : MODERATE EFFORT (3-5 Personnel, Standard Patrol)
Barricades     : No static barricades required


In [15]:
# CELL 9: Exporting Artifacts for Streamlit App
import joblib

# Save Models
model_tii.save_model('lightgbm_tii_model.txt')
model_closure.save_model('lightgbm_closure_model.txt')

# Save NLP Artifacts
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
joblib.dump(svd, 'svd_transformer.pkl')

# Extract Adjacency & Duration Database for the UI to query
historic_db = df[['zone', 'corridor', 'junction', 'event_cause', 'duration_mins']].copy()
historic_db.to_csv('historic_database.csv', index=False)

print("✅ All Models, NLP artifacts, and Historic Databases successfully saved!")

✅ All Models, NLP artifacts, and Historic Databases successfully saved!


In [2]:
# STANDALONE LATENCY BENCHMARKING
import time
import pandas as pd
import lightgbm as lgb
import joblib

print("Loading System Artifacts into RAM...")

# 1. Load the models and transformers (Assuming Cell 9 was previously run)
try:
    model_tii = lgb.Booster(model_file='lightgbm_tii_model.txt')
    model_closure = lgb.Booster(model_file='lightgbm_closure_model.txt')
    tfidf = joblib.load('tfidf_vectorizer.pkl')
    svd = joblib.load('svd_transformer.pkl')
    print("✅ Artifacts Loaded Successfully.\n")
except Exception as e:
    print(f"❌ Error loading files: {e}. Please run the export cell first.")
    exit()

print("Benchmarking Edge Inference Speed...")

# Simulate a new incoming dispatch call
sample_text = "Accident on the main road, heavy truck flipped over blocking two lanes."

# --- WARMUP RUN ---
# (Wakes up the CPU cache and library memory allocations to ensure an accurate test)
_ = tfidf.transform(["warmup text"]) 

# ==========================================
# ⏱️ START STOPWATCH
# ==========================================
start_time = time.time()

# 1. NLP Processing
tfidf_vec = tfidf.transform([sample_text.lower()])
semantic_feats = svd.transform(tfidf_vec)[0]

# 2. Build the Input Row
input_data = pd.DataFrame({
    'event_cause': ['accident'],
    'corridor': ['Tumkur Road'],
    'zone': ['West Zone 1'],
    'hour': [9],
    'day_of_week': [0],
    'is_weekend': [0],
    'veh_bucket': ['large_vehicle'],
    'concurrent_events': [2],
    'semantic_f0': [semantic_feats[0]],
    'semantic_f1': [semantic_feats[1]],
    'semantic_f2': [semantic_feats[2]],
    'semantic_f3': [semantic_feats[3]],
    'semantic_f4': [semantic_feats[4]]
})

for col in ['event_cause', 'corridor', 'zone', 'veh_bucket']:
    input_data[col] = input_data[col].astype('category')

# 3. Dual AI Inference
raw_tii = model_tii.predict(input_data)[0]
closure_prob = model_closure.predict(input_data)[0]

# ==========================================
# 🛑 STOP STOPWATCH
# ==========================================
end_time = time.time()

latency_ms = (end_time - start_time) * 1000

print("-" * 50)
print(f"Predicted TII:      {raw_tii:.1f} / 100")
print(f"Closure Risk:       {closure_prob*100:.1f}%")
print("-" * 50)
print(f"🚀 TOTAL INFERENCE LATENCY: {latency_ms:.2f} milliseconds")
print("-" * 50)

Loading System Artifacts into RAM...
✅ Artifacts Loaded Successfully.

Benchmarking Edge Inference Speed...
--------------------------------------------------
Predicted TII:      95.5 / 100
Closure Risk:       2.5%
--------------------------------------------------
🚀 TOTAL INFERENCE LATENCY: 26.53 milliseconds
--------------------------------------------------
